# Recommendation

# Step 1: Load the Dataset


In [1]:
import pandas as pd

# Load the dataset
# file_path = "recipe_final (1).csv"
file_path = "recipes_kaggle_images.csv"
recipe_df = pd.read_csv(file_path)

recipe_df.head()

,Unnamed: 0,recipe_id,recipe_name,aver_rate,image_url,review_nums,calories,fat,carbohydrates,protein,cholesterol,sodium,fiber,ingredients_list
0,0,222388,Homemade Bacon,5.00,/kaggle/input/foodrecsysv1/raw-data-images/raw...,3,15,36,1,42,21,81,2,"['pork belly', 'smoked paprika', 'kosher salt'..."
1,1,240488,"Pork Loin, Apples, and Sauerkraut",4.76,/kaggle/input/foodrecsysv1/raw-data-images/raw...,29,19,18,10,73,33,104,41,"['sauerkraut drained', 'Granny Smith apples sl..."
2,2,218939,Foolproof Rosemary Chicken Wings,4.57,/kaggle/input/foodrecsysv1/raw-data-images/raw...,12,17,36,2,48,24,31,4,"['chicken wings', 'sprigs rosemary', 'head gar..."
3,3,87211,Chicken Pesto Paninis,4.62,/kaggle/input/foodrecsysv1/raw-data-images/raw...,163,32,45,20,65,20,43,18,"['focaccia bread quartered', 'prepared basil p..."
4,4,245714,Potato Bacon Pizza,4.50,/kaggle/input/foodrecsysv1/raw-data-images/raw...,2,8,12,5,14,7,8,3,"['red potatoes', 'strips bacon', 'Sauce:', 'he..."


In [2]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

In [3]:
recipe_df["ingredients_list"].head()

0    ['pork belly', 'smoked paprika', 'kosher salt'...
1    ['sauerkraut drained', 'Granny Smith apples sl...
2    ['chicken wings', 'sprigs rosemary', 'head gar...
3    ['focaccia bread quartered', 'prepared basil p...
4    ['red potatoes', 'strips bacon', 'Sauce:', 'he...
Name: ingredients_list, dtype: object

In [8]:
# Preprocess Ingredients
vectorizer = TfidfVectorizer(max_features=500)
X_ingredients = vectorizer.fit_transform(recipe_df["ingredients_list"])

In [9]:
# Normalize Numerical Features
scaler = StandardScaler()
X_numerical = scaler.fit_transform(
    recipe_df[
        [
            "calories",
            "fat",
            "carbohydrates",
            "protein",
            "cholesterol",
            "sodium",
            "fiber",
        ]
    ]
)

In [10]:
# Combine Features
X_combined = np.hstack([X_numerical, X_ingredients.toarray()])

# Train KNN Model
knn = NearestNeighbors(n_neighbors=3, metric="euclidean")
knn.fit(X_combined)

NearestNeighbors(metric='euclidean', n_neighbors=3)

In [11]:
# Function to Recommend Recipes
def recommend_recipes(input_features):
    input_features_scaled = scaler.transform([input_features[:7]])
    input_ingredients_transformed = vectorizer.transform([input_features[7]])
    input_combined = np.hstack(
        [input_features_scaled, input_ingredients_transformed.toarray()]
    )
    distances, indices = knn.kneighbors(input_combined)
    recommendations = recipe_df.iloc[indices[0]]
    return recommendations[["recipe_name", "ingredients_list", "image_url"]]


# Example Input
input_features = [15, 36, 1, 42, 21, 81, 2, "pork belly, smoked paprika, kosher salt"]
recommendations = recommend_recipes(input_features)
recommendations

/home/galih/DCML/.env/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,recipe_name,ingredients_list,image_url
0,Homemade Bacon,"['pork belly', 'smoked paprika', 'kosher salt'...",/kaggle/input/foodrecsysv1/raw-data-images/raw...
9975,Pork Carnitas,"['vegetable oil', 'pork shoulder', 'kosher sal...",/kaggle/input/foodrecsysv1/raw-data-images/raw...
10708,Picnic Sausage,"['ground beef', 'ground spicy pork sausage', '...",/kaggle/input/foodrecsysv1/raw-data-images/raw...
